# 12_gold_backfill
**Run this once**, right after the Gold tables exist, to build the Gold history.
Silver only keeps the current state of each game, so the daily history (one fact row per game per day it changed) can only be rebuilt from Bronze. This notebook empties Silver and the fact table, then replays every Bronze date in order: `03_bronze_to_silver`, then `10_silver_to_gold`. Bronze is never touched. It follows the same approach as `07_rebuild_silver`.
Set the `confirm` widget to `YES` to run.

In [0]:
CONFIRM = "YES"
if CONFIRM != "YES":
    raise ValueError("Set CONFIRM to YES to empty Silver and the fact table and replay all dates.")

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("confirm", "YES")
print("confirm =", dbutils.widgets.get("confirm"))

In [0]:
spark.sql("USE workspace.steam")
silver_before = spark.table("silver_steam").count()
dates = [str(r.load_date) for r in spark.sql("""
    SELECT DISTINCT load_date FROM bronze_steam
    WHERE load_date >= DATE'2026-10-09'
    ORDER BY load_date
""").collect()]
print("silver rows before:", silver_before)
print("dates to replay:", dates)

In [0]:
spark.sql("DELETE FROM silver_steam")
spark.sql("DELETE FROM fact_game_daily")
print("silver and fact emptied")

for d in dates:
    dbutils.notebook.run("./03_bronze_to_silver", 3600, {"date": d, "batch_id": ""})
    dbutils.notebook.run("./10_silver_to_gold", 3600, {"date": d})
    print(d, "done | silver:", spark.table("silver_steam").count(), "| fact:", spark.table("fact_game_daily").count())

In [0]:
silver_after = spark.table("silver_steam").count()
print(f"silver rows: {silver_before} before, {silver_after} after (should be equal)")
display(spark.sql("""
    SELECT date_key, COUNT(*) AS fact_rows, SUM(ccu) AS total_ccu
    FROM fact_game_daily GROUP BY date_key ORDER BY date_key
"""))